## iterativePCE: alpha-annealed Pauli Correlation Encoding

PCE relaxes the sign function used to decide each binary variable with `tanh(alpha * <Pi_i>)`, a smooth surrogate controlled by a sharpness parameter `alpha`. A single fixed `alpha` faces a tradeoff: too small and the variables never really binarize (the decision is "unconfident"); too large and `tanh`'s derivative vanishes almost everywhere, stalling the optimizer. Even the library's own default heuristic for `alpha` is just one fixed guess, picked once before any optimization happens.

`iterativePCE` avoids having to guess a single good `alpha` up front: it starts small and, every round, nudges `alpha` up just enough to push the *least* binarized correlator past a threshold `M`, warm-starting the ansatz from the previous round's optimum. It wraps `PCE` (or any future class sharing the same API) by composition -- it doesn't subclass it -- so it re-runs the wrapped algorithm to convergence once per round.

This implements Algorithm 1 ("Iterative-alpha PCE heuristic") from https://arxiv.org/abs/2602.17479.

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

from qarp import config
from qarp.algorithms import PCE, iterativePCE
from qarp.algorithms import calculate_qubits, classical_function_max_cut
from qarp.algorithms import StateVector
from qarp.blocks import HEABlock
from qarp.optimizers import ScipyOptimizer

config.seed = 1234
np.random.seed(config.seed)

### Problem setup

We use the same size as the smallest instance in the paper: 6 nodes, order-2 correlators, 3 qubits.

In [ ]:
n_nodes = 10
order = 2

graph = nx.gnm_random_graph(n_nodes, 10, seed=config.seed)
for u, v in graph.edges:
    graph[u][v]["weight"] = 1.0

n_qubits = calculate_qubits(n_nodes, order)
print("Number of qubits:", n_qubits)

he_wfn = HEABlock(n_qubits, 2, True, True, True, False).build()
initial_parameters = [0.1] * len(list(he_wfn.symbols))

### `iterativePCE`: adaptively annealing `alpha`

Starting from `alpha0 = 1.0`, `iterativePCE` repeatedly re-runs `PCE`, each time rescaling `alpha` to push the least-binarized correlator just past `threshold = 0.9`, until every correlator is binarized, without ever having to guess a single good `alpha` up front.

In [ ]:
ipce = iterativePCE(
    graph,
    order,
    he_wfn,
    alpha0=1.0,
    threshold=0.9,
    max_outer_iterations=40,
    primitive=StateVector(),
    initial_parameters=initial_parameters,
    verbose=False,
    optimizer=ScipyOptimizer("COBYQA"),
).build()

_, x_iterative, solution_iterative = ipce.run()

print("Rounds run:", len(ipce.alpha_history))
print("Final alpha:", ipce.alpha_history[-1])
print("Iterative solution:", solution_iterative)
print("Iterative cut size:", classical_function_max_cut(graph, solution_iterative))

### Convergence across rounds

In [ ]:
binarization_per_round = [
    float(np.mean(np.abs(np.tanh(a * r)) >= ipce.threshold))
    for a, r in zip(ipce.alpha_history, ipce.raw_expectations_history)
]
cut_size_per_round = [
    classical_function_max_cut(graph, sol) for sol in ipce.solution_history
]

plt.figure(figsize=(12, 4))
plt.subplot(131)
plt.title("alpha per round")
plt.plot(ipce.alpha_history, marker="o")
plt.yscale("log")
plt.xlabel("round")
plt.ylabel("alpha")
plt.grid()
plt.subplot(132)
plt.title("Binarized fraction per round")
plt.plot(binarization_per_round, marker="o")
plt.axhline(1.0, color="gray", linestyle="--", linewidth=1)
plt.xlabel("round")
plt.ylabel("fraction binarized")
plt.ylim(-0.05, 1.05)
plt.grid()
plt.subplot(133)
plt.title("Cut size per round")
plt.plot(cut_size_per_round, marker="o")
plt.xlabel("round")
plt.ylabel("cut size")
plt.grid()
plt.tight_layout()
plt.show()

By construction, `iterativePCE` keeps annealing `alpha` until *every* correlator clears the binarization threshold, or until `max_outer_iterations` is reached -- something a single fixed `alpha` has no mechanism to do at all. The plots above show this directly: `alpha` grows monotonically round over round, and the binarized fraction climbs toward 1.0 as fewer and fewer correlators remain undecided.